In [5]:
# ---File Paths & Config--------------------------------------------------------
WEIGHTS = r"C:\Users\T\Documents\WildScan\demo\wildscan_best_scratch.pt"
CLASSES = r"C:\Users\T\Documents\WildScan\demo\classes.json"
DETECTOR_WEIGHTS = r"C:\Users\T\Documents\WildScan\demo\md_v5a.0.0.pt"
DET_CONF  = 0.15
DET_IMGSZ = 1280

import sys, types
def _ensure_hf_compat():
    try:
        import huggingface_hub.utils._errors as _; import huggingface_hub.utils._validators as _
        return
    except Exception:
        pass
    try:
        from huggingface_hub import errors as _errors_mod 
        RepoNotFound = getattr(_errors_mod, "RepositoryNotFoundError", Exception)
        HFValidErr   = getattr(_errors_mod, "HFValidationError", Exception)
    except Exception:
        class RepoNotFound(Exception): ...
        class HFValidErr(Exception): ...
    if "huggingface_hub.utils" not in sys.modules:
        sys.modules["huggingface_hub.utils"] = types.ModuleType("huggingface_hub.utils")

    err_mod = types.ModuleType("huggingface_hub.utils._errors")
    err_mod.RepositoryNotFoundError = RepoNotFound
    err_mod.HFValidationError = HFValidErr
    sys.modules["huggingface_hub.utils._errors"] = err_mod

    val_mod = types.ModuleType("huggingface_hub.utils._validators")
    val_mod.HFValidationError = HFValidErr
    sys.modules["huggingface_hub.utils._validators"] = val_mod

_ensure_hf_compat()

In [6]:
import json, torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms
from PIL import Image, ImageDraw
import gradio as gr

yolo_model = None
ANIMAL_IDS = set()
if DETECTOR_WEIGHTS:
    import yolov5
    yolo_model = yolov5.load(DETECTOR_WEIGHTS)
    yolo_model.conf = DET_CONF
    yolo_model.iou  = 0.45
    yolo_model.max_det = 50
    names = yolo_model.names
    if isinstance(names, dict):
        ANIMAL_IDS = {i for i, n in names.items() if str(n).lower() == "animal"} or {0}
    else:
        ANIMAL_IDS = {i for i, n in enumerate(names) if str(n).lower() == "animal"} or {0}

class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_c)
        self.down  = downsample
    def forward(self, x):
        idt = x
        out = F.relu(self.bn1(self.conv1(x))); out = self.bn2(self.conv2(out))
        if self.down is not None: idt = self.down(x)
        return F.relu(out + idt)

class ScratchResNet(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(3, 2, 1)
        )
        self.layer1 = self._make_layer(64,  64, 2, 1)
        self.layer2 = self._make_layer(64, 128, 2, 2)
        self.layer3 = self._make_layer(128,256,2, 2)
        self.layer4 = self._make_layer(256,512,2, 2)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Linear(512, num_classes)
    def _make_layer(self, in_c, out_c, blocks, stride):
        down = None
        if stride != 1 or in_c != out_c:
            down = nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride, bias=False),
                                 nn.BatchNorm2d(out_c))
        layers = [ResidualBlock(in_c, out_c, stride, down)]
        for _ in range(1, blocks): layers.append(ResidualBlock(out_c, out_c))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.stem(x)
        for l in (self.layer1,self.layer2,self.layer3,self.layer4): x = l(x)
        x = self.avg(x).flatten(1)
        return self.fc(x)

with open(CLASSES, "r", encoding="utf-8") as f:
    data = json.load(f)
classes = data["idx_to_name"] if isinstance(data, dict) and "idx_to_name" in data else data
if isinstance(classes, dict):
    classes = [classes[str(i)] for i in range(len(classes))]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load checkpoint first (and quiet the warning)
state = torch.load(WEIGHTS, map_location=device, weights_only=True)  # PyTorch 2.4+ supports weights_only
if isinstance(state, dict) and "model_state" in state:
    state = state["model_state"]

# infer num_classes from the fc layer in the checkpoint
num_classes_ckpt = state["fc.weight"].shape[0]

# build model with the right head size
model = ScratchResNet(num_classes=num_classes_ckpt).to(device).eval()
model.load_state_dict(state, strict=True)

# ensure your classes list matches that size; trim if needed
if len(classes) != num_classes_ckpt:
    classes = classes[:num_classes_ckpt]


tfm = transforms.Compose([
    transforms.Resize(288),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225]),
])

def classify_pil(pil_img, topk=5):
    x = tfm(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = model(x).softmax(1)[0]
        conf, idx = probs.topk(topk)
    labels = [classes[i] for i in idx.tolist()]
    confs  = [float(c) for c in conf.tolist()]
    return labels, confs

def detect_boxes(pil_img):
    W, H = pil_img.size
    if yolo_model is None:
        return [[0, 0, W, H]]
    res = yolo_model(pil_img, size=DET_IMGSZ) 
    boxes = []
    xyxy = res.xyxy[0].cpu().numpy() if hasattr(res, "xyxy") else res.pred[0].cpu().numpy()
    for x1, y1, x2, y2, conf, cls_id in xyxy:
        cls_id = int(cls_id)
        if ANIMAL_IDS and cls_id not in ANIMAL_IDS:
            continue
        x1 = max(0, min(int(x1), W)); x2 = max(0, min(int(x2), W))
        y1 = max(0, min(int(y1), H)); y2 = max(0, min(int(y2), H))
        if x2 > x1 and y2 > y1:
            boxes.append([x1, y1, x2, y2])
    return boxes or [[0, 0, W, H]]

def infer(img, topk=5, det_conf=0.15, box_hex="#38BDF8"):
    # UI settings
    BOX_COLOR  = (56, 189, 248, 255) 
    TEXT_BG    = (2, 6, 23, 200)
    TEXT_COLOR = (255, 255, 255, 255)

    if yolo_model is not None:
        yolo_model.conf = float(det_conf)

    h = box_hex.lstrip("#")
    rgb = tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
    box_col = (*rgb, 255)

    pil = Image.fromarray(img).convert("RGB")
    draw = ImageDraw.Draw(pil, "RGBA")
    boxes = detect_boxes(pil)

    agg = {}
    for (x1, y1, x2, y2) in boxes:
        crop = Image.fromarray(img[y1:y2, x1:x2, :]).convert("RGB")
        labels, confs = classify_pil(crop, topk=topk)
        draw.rectangle([x1, y1, x2, y2], outline=box_col, width=3)
        tag = f"{labels[0]} {confs[0]:.2f}"
        tw = draw.textlength(tag)
        draw.rectangle([x1, y1-18, x1+tw+10, y1], fill=TEXT_BG)
        draw.text((x1+5, y1-16), tag, fill=TEXT_COLOR)
        for l, c in zip(labels, confs):
            agg[l] = max(agg.get(l, 0.0), c)

    agg_sorted = dict(sorted(agg.items(), key=lambda kv: kv[1], reverse=True)[:topk])
    return pil, agg_sorted

# Interface
gr.Interface(
    fn=infer,
    inputs=gr.Image(type="numpy", label="Upload an image"),
    outputs=[
        gr.Image(type="pil", label="Annotated result"),
        gr.Label(num_top_classes=5, label="Predictions")
    ],
    title="WildScan Demo",
    description=(
        "Upload any wildlife image. We detect animals!"
        "<br><br><small>Developed by Tyler Clinscales, Geoffrey Fadera, and Edwin Merchan. University of San Diego — WildScan"
    ),
    allow_flagging="manual",         
    flagging_options=["Misclassified", "Low Confidence", "Other"],
    flagging_dir="flagged",
).launch()

YOLOv5  2025-8-10 Python-3.10.18 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)

C:\Users\T\Miniconda3\envs\pytorch_wildlife\lib\site-packages\yolov5\models\experimental.py:79: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an i

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


C:\Users\T\Miniconda3\envs\pytorch_wildlife\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\T\Miniconda3\envs\pytorch_wildlife\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\T\Miniconda3\envs\pytorch_wildlife\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\T\Miniconda3\envs\pytorch_wildlife\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\T\Miniconda

Created dataset file at: flagged\dataset1.csv


C:\Users\T\Miniconda3\envs\pytorch_wildlife\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
